# Notebook 01 · Datos y exploración del corpus

**Proyecto:** Análisis de sentimiento y de dimensiones semánticas de la experiencia del huésped en reseñas de hoteles de lujo
**Autores:** Isabel Cristina Morales Parra – Jaime Esteban Bejarano Caldas
**Curso:** Procesamiento de Lenguaje Natural · Maestría en Analítica Aplicada e Inteligencia Artificial, Universidad de La Sabana · Profesor: Andrés Rosso

**Objetivos que cubre este notebook:** objetivo específico 1 (caracterización del corpus mediante análisis exploratorio textual) y objetivo específico 2 (normalización y limpieza del texto).

## 0. Verificación del entorno

Se comprueba que el entorno de trabajo `nlp312` tenga instaladas las librerías del proyecto:
- `pandas` y `numpy`: manipulación de datos.
- `scikit-learn`: representaciones dispersas, clasificadores y modelado de tópicos.
- `gensim`: Word2Vec.
- `nltk` y `spacy`: normalización del texto.
- `seaborn`, `plotly` y `wordcloud`: visualización.

In [1]:
import sys
import importlib

print("Versión de Python:", sys.version.split()[0])
print("Ejecutable:", sys.executable)

paquetes = ["pandas", "numpy", "sklearn", "gensim", "nltk", "spacy",
            "matplotlib", "seaborn", "plotly", "wordcloud"]

for nombre in paquetes:
    try:
        modulo = importlib.import_module(nombre)
        version = getattr(modulo, "__version__", "sin versión")
        print(f"{nombre:<11} OK      {version}")
    except ImportError:
        print(f"{nombre:<11} FALTA")

Versión de Python: 3.12.16
Ejecutable: c:\Users\isabe\.conda\envs\nlp312\python.exe
pandas      OK      3.1.0
numpy       OK      2.5.4
sklearn     OK      1.9.1
gensim      OK      4.4.0
nltk        OK      3.10.4
spacy       OK      3.8.18
matplotlib  OK      3.11.2
seaborn     OK      0.13.2
plotly      OK      7.1.0
wordcloud   OK      1.9.6


## 1. Carga de los datos

Se cargan las librerías de uso general y el archivo `LuxuryHotels.csv` desde `data/raw/`. Las rutas se construyen a partir de la raíz del repositorio, para que el notebook funcione en cualquier computador que clone el proyecto, sin modificar rutas absolutas. Se fija una semilla aleatoria (`SEMILLA = 42`) para todos los procesos con componente aleatorio (particiones, Word2Vec, modelos de tópicos), lo que garantiza la reproducibilidad de los resultados.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px

# --- Rutas relativas a la raíz del repositorio ---
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_DATOS = RAIZ / "data" / "raw" / "LuxuryHotels.csv"
RUTA_FIGURAS = RAIZ / "figures"
RUTA_RESULTADOS = RAIZ / "results"

# --- Configuración general ---
SEMILLA = 42
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", palette="deep")

# --- Carga ---
df_raw = pd.read_csv(RUTA_DATOS)

print("Raíz del proyecto:", RAIZ)
print("Dimensiones:", df_raw.shape)
df_raw.info()
df_raw.head()

Raíz del proyecto: c:\Users\isabe\Cursos\ip_2026_b\NLP\pln-luxuryhotels-sentimiento
Dimensiones: (18074, 4)
<class 'pandas.DataFrame'>
RangeIndex: 18074 entries, 0 to 18073
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Bubbles  18074 non-null  str  
 1   Review   18074 non-null  str  
 2   Rating   18074 non-null  int64
 3   YY       18074 non-null  int64
dtypes: int64(2), str(2)
memory usage: 564.9 KB


,Bubbles,Review,Rating,YY
0,5.0 of 5 bubbles,"It was late evening when we , a family of 3 , checked-in . Front office had 3 persons manning the counters, ,2 girls...",5,22
1,5.0 of 5 bubbles,"We had a very nice stay at Akama . Went here with my mom , dad and grand mom for Pashupatinath Darshan . They rooms ...",5,22
2,5.0 of 5 bubbles,"Hey you! Yes, you! If you are struggling to choose a hotel a hotel in Kathmandu (like I did), then you should absolu...",5,23
3,1.0 of 5 bubbles,"Hello Everyone,If you are planning to visit this hotel , please never think about it.. there is many many best hotel...",1,21
4,5.0 of 5 bubbles,"The stay is very very comfortable. All rooms are very hygienic, tidy, spacious and the staffs are very courteous and...",5,23


La inspección de las primeras filas anticipa dos rasgos del texto que se tratarán en la normalización: (1) **puntuación sin espacio posterior** (por ejemplo, «Hello Everyone,If you»), que un tokenizador simple uniría en una sola palabra («everyone,if»), y (2) **espacios irregulares antes de la puntuación** («we , a family», «counters, ,2»). También aparecen nombres propios de hoteles y lugares (Akama, Kathmandu, Pashupatinath), lo que confirma la concentración geográfica del corpus en Nepal.

## 2. Auditoría de calidad de los datos

Antes de modificar el corpus se cuantifican sus problemas de calidad. Este bloque solo diagnostica; las decisiones de depuración se toman y justifican en el bloque siguiente. Se verifica:
1. Si `Bubbles` es redundante con `Rating`.
2. Cuántos textos están duplicados.
3. Si hay textos repetidos con calificaciones contradictorias.
4. Cuántas reseñas son demasiado cortas para aportar información.
5. Cuántas reseñas están escritas en un sistema de escritura no latino (japonés, chino, etc.), lo que indica que no están en inglés.

In [3]:
auditoria = {}

# 1. ¿Bubbles es redundante con Rating?
bubbles_num = df_raw["Bubbles"].str.extract(r"(\d)\.0 of 5")[0].astype(int)
auditoria["Bubbles coincide con Rating (%)"] = round((bubbles_num == df_raw["Rating"]).mean() * 100, 2)

# 2. Duplicados exactos del texto
auditoria["Textos duplicados (copias adicionales)"] = int(df_raw["Review"].duplicated().sum())

# 3. Textos repetidos con calificaciones distintas
calificaciones_por_texto = df_raw.groupby("Review")["Rating"].nunique()
textos_conflictivos = calificaciones_por_texto[calificaciones_por_texto > 1].index
auditoria["Textos con calificaciones contradictorias"] = len(textos_conflictivos)

# 4. Reseñas muy cortas
n_palabras = df_raw["Review"].str.split().str.len()
auditoria["Reseñas con menos de 10 palabras"] = int((n_palabras < 10).sum())

# 5. Sistema de escritura: proporción de letras latinas (a-z) sobre el total de letras
letras_totales = df_raw["Review"].str.count(r"[^\W\d_]")
letras_latinas = df_raw["Review"].str.count(r"[A-Za-z]")
prop_latina = letras_latinas / letras_totales.clip(lower=1)
auditoria["Reseñas con menos del 50 % de letras latinas"] = int((prop_latina < 0.5).sum())

display(pd.Series(auditoria, name="Valor").to_frame())

print("\nTextos con calificaciones contradictorias:")
display(df_raw.loc[df_raw["Review"].isin(textos_conflictivos), ["Review", "Rating", "YY"]])

print("\nEjemplos de reseñas en escritura no latina:")
display(df_raw.loc[prop_latina < 0.5, ["Review", "Rating"]].head(5))

,Valor
Bubbles coincide con Rating (%),100.0
Textos duplicados (copias adicionales),97.0
Textos con calificaciones contradictorias,1.0
Reseñas con menos de 10 palabras,123.0
Reseñas con menos del 50 % de letras latinas,163.0



Textos con calificaciones contradictorias:


,Review,Rating,YY
9256,#NAME?,1,19
10637,#NAME?,4,17
17034,#NAME?,5,23



Ejemplos de reseñas en escritura no latina:


,Review,Rating
1233,６月４日～５日に宿泊しました。正直、古い感じが印象的でした。昔は、カトマンズを代表するホテルだったと思わせるような作りですが、好きな人は好きかもしれません。部屋もやはり古い感じでした。朝食はメニューが豊富でしたが、正直、味は・・・。,3
1315,50年の歴史がある五つ星ということで期待していましたが、微妙なところですね。WiFiも有料の割には使いにくいし、風呂の水も茶色いし、ベッドも小さいし、もちろん良いところもあるのでしょうけど、値段の割にはと思うところが目についてしまい...,3
1360,ホテルスタッフやドアマンは笑顔で挨拶してくれます。部屋も清潔です。ホテル内の中華レストランを利用しましたが美味しかったです。タメル地区やダルバール広場へも徒歩で可能です。ホテルの近くに日本料理のレストランがあります。,4
1480,ご存知の通り、カトマンズは安宿がたくさんあり、そこに泊まることが旅の醍醐味でもあるとおもいます。その中で、このホテルは、繁華街から歩いて5分くらいのところにあります。停電にもすぐに自家発電が付き、暗くなるのは30秒程度です。設備など...,4
1548,"Нас сюда поселила авиакомпания, когда была задержка рейса. Мне повезло, попался номер люкс. Ну все супер просто. И ч...",5


**Resultado de la auditoría.**

| Hallazgo | Valor | Decisión |
|:--|--:|:--|
| `Bubbles` coincide con `Rating` | 100 % | Se descarta `Bubbles` por redundante |
| Textos duplicados | 97 copias | Se conserva una sola ocurrencia de cada texto |
| Texto con calificaciones contradictorias | 1 | Se eliminan sus 3 registros (ver abajo) |
| Reseñas con menos de 10 palabras | 123 | Se conservan: textos breves como «Excellent stay» expresan polaridad |
| Reseñas en escritura no latina | 163 | Indicio de reseñas en otros idiomas; se cuantifica en el bloque 3 |

**Anomalía: el texto `#NAME?`.** El único texto con calificaciones contradictorias (1, 4 y 5) no es una reseña, sino un **error de exportación desde Excel**: Excel muestra `#NAME?` cuando interpreta como fórmula una celda que empieza por `=`, `+`, `-` o `@`. El contenido original de esas tres reseñas se perdió. Por eso se eliminan los tres registros, en lugar de conservar una ocurrencia como se propuso en el anteproyecto: no contienen texto que analizar.

**Anomalía: corpus multilingüe.** El corpus no está completamente en inglés, como se asumió en el anteproyecto. Aparecen reseñas en japonés, chino y ruso, y el criterio de alfabeto no detecta los idiomas que usan letras latinas (español, francés, turco). Se requiere un detector de idioma.

## 3. Detección de idioma

Se identifica el idioma de cada reseña con `langdetect`, una adaptación en Python de la librería *language-detection* de Google. El detector compara la frecuencia de secuencias de caracteres (n-gramas de caracteres) del texto con perfiles estadísticos de 55 idiomas. Como su algoritmo tiene un componente aleatorio, se fija una semilla para que el resultado sea reproducible.

Antes de la detección se eliminan los registros `#NAME?` y los duplicados, para no clasificar textos que se van a descartar.

In [4]:
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = SEMILLA  # langdetect es aleatorio: la semilla lo hace reproducible

def detectar_idioma(texto):
    """Devuelve el código ISO del idioma del texto, o 'desconocido' si no se puede detectar."""
    try:
        return detect(texto)
    except LangDetectException:
        return "desconocido"

# Depuración previa: errores de exportación y duplicados
df = df_raw[df_raw["Review"].str.strip() != "#NAME?"].drop_duplicates(subset="Review").copy()
print("Registros tras eliminar #NAME? y duplicados:", len(df))

df["idioma"] = df["Review"].apply(detectar_idioma)

conteo_idiomas = df["idioma"].value_counts()
print("Reseñas en inglés:", conteo_idiomas.get("en", 0))
print("Reseñas en otros idiomas:", (df["idioma"] != "en").sum(), "en", conteo_idiomas.size - 1, "idiomas")

fig = px.bar(conteo_idiomas.drop("en").head(12).reset_index(), x="idioma", y="count",
             title="Reseñas que no están en inglés, por idioma (12 más frecuentes)",
             labels={"idioma": "Idioma (código ISO 639-1)", "count": "Reseñas"})
fig.show()

print("\nCalificación de las reseñas que no están en inglés:")
display(df.loc[df["idioma"] != "en", "Rating"].value_counts().sort_index().to_frame())

Registros tras eliminar #NAME? y duplicados: 17976
Reseñas en inglés: 17263
Reseñas en otros idiomas: 713 en 23 idiomas



Calificación de las reseñas que no están en inglés:


,count
Rating,
1,9
2,22
3,77
4,246
5,359


**Resultado.** De las 17.976 reseñas únicas válidas, **17.263 (96,0 %) están en inglés** y **713 (4,0 %) en otros 23 idiomas**. Los más frecuentes son español (161), italiano (110), japonés (102), alemán (92) y francés (83). Esta composición es coherente con un destino turístico internacional como Nepal.

**Decisión: se conservan solo las reseñas en inglés.** Las herramientas del proyecto son monolingües: palabras vacías, lematizador `en_core_web_sm`, Word2Vec entrenado sobre el corpus y palabras ancla de las proyecciones semánticas. Una reseña en otro idioma entraría al modelo como un conjunto de palabras raras que solo agregan ruido, y sus términos dispersarían el vocabulario.

**Costo de la decisión.** Se pierden 31 reseñas negativas (9 con calificación 1 y 22 con calificación 2), que representan cerca del 4,5 % de la clase minoritaria del escenario A. Es una pérdida pequeña, pero se registra porque esa clase es la más escasa.

**Limitaciones del detector.**
- `langdetect` es menos confiable con textos muy cortos o mixtos. Por ejemplo, una reseña en hindi escrita con letras latinas («hotel staff bahot achha hai») se clasifica como indonesio.
- Pese a la semilla fija, el resultado puede variar en algún caso límite entre equipos: en una ejecución exploratoria se obtuvieron 17.262 reseñas en inglés, una menos que aquí.

**Trabajo futuro:** traducir las reseñas o usar modelos multilingües (por ejemplo, *embeddings* multilingües) para aprovechar este 4 % del corpus.

## 4. Corpus de trabajo y variable objetivo

Se construye el corpus definitivo (reseñas únicas, válidas y en inglés) y se documenta cuántos registros elimina cada paso de depuración (trazabilidad). Luego se crean las tres definiciones de la variable objetivo propuestas en el anteproyecto:

| Escenario | Definición | Uso |
|:--|:--|:--|
| **A. Binario polar** | positivo = 4–5; negativo = 1–2; se excluye 3 | Principal |
| **B. Tres clases** | negativo = 1–2; neutral = 3; positivo = 4–5 | Complementario |
| **C. Satisfacción plena** | excelente = 5; no excelente = 1–4 | Alternativo |

El corpus depurado se guarda en `data/processed/` para que los notebooks siguientes partan de los mismos datos.

In [5]:
# --- Corpus definitivo ---
df_en = df[df["idioma"] == "en"].drop(columns=["Bubbles", "idioma"]).reset_index(drop=True)
df_en["anio"] = 2000 + df_en["YY"]              # supuesto: YY = año en dos dígitos
df_en["n_palabras"] = df_en["Review"].str.split().str.len()

# --- Variable objetivo: tres escenarios ---
df_en["polaridad_A"] = df_en["Rating"].map({1: "negativo", 2: "negativo", 4: "positivo", 5: "positivo"})  # 3 queda vacío (NaN)
df_en["clase_B"] = pd.cut(df_en["Rating"], bins=[0, 2, 3, 5], labels=["negativo", "neutral", "positivo"])
df_en["excelente_C"] = np.where(df_en["Rating"] == 5, "excelente", "no excelente")

# --- Trazabilidad de la depuración ---
trazabilidad = pd.DataFrame({
    "Paso": ["Archivo original", "Sin registros #NAME?", "Sin duplicados", "Solo inglés"],
    "Registros": [len(df_raw),
                  (df_raw["Review"].str.strip() != "#NAME?").sum(),
                  len(df),
                  len(df_en)],
})
trazabilidad["Eliminados en el paso"] = (-trazabilidad["Registros"].diff()).fillna(0).astype(int)
display(trazabilidad)

# --- Distribución de clases en cada escenario ---
for columna in ["polaridad_A", "clase_B", "excelente_C"]:
    dist = df_en[columna].value_counts().to_frame("Reseñas")
    dist["%"] = (dist["Reseñas"] / dist["Reseñas"].sum() * 100).round(1)
    print(f"\n{columna}  (n = {dist['Reseñas'].sum():,})")
    display(dist)

# --- Guardar resultados ---
df_en.to_csv(RAIZ / "data" / "processed" / "resenas_en_depuradas.csv", index=False)
trazabilidad.to_csv(RUTA_RESULTADOS / "01_trazabilidad_depuracion.csv", index=False)
print("Archivos guardados en data/processed/ y results/")

,Paso,Registros,Eliminados en el paso
0,Archivo original,18074,0
1,Sin registros #NAME?,18071,3
2,Sin duplicados,17976,95
3,Solo inglés,17263,713



polaridad_A  (n = 16,438)


,Reseñas,%
polaridad_A,,
positivo,15788,96.0
negativo,650,4.0



clase_B  (n = 17,263)


,Reseñas,%
clase_B,,
positivo,15788,91.5
neutral,825,4.8
negativo,650,3.8



excelente_C  (n = 17,263)


,Reseñas,%
excelente_C,,
excelente,13154,76.2
no excelente,4109,23.8


Archivos guardados en data/processed/ y results/


**Resultado.** El corpus de trabajo queda con **17.263 reseñas únicas en inglés** (95,5 % del archivo original). La depuración eliminó 811 registros: 3 errores de exportación (`#NAME?`), 95 duplicados y 713 reseñas en otros idiomas. Los 97 duplicados de la auditoría incluían 2 copias de `#NAME?`, que ya se habían eliminado en el paso anterior.

| Escenario | n | Clase minoritaria | Razón de desbalance |
|:--|--:|:--|--:|
| A. Binario polar | 16.438 | negativo: 650 (4,0 %) | ≈ 24 : 1 |
| B. Tres clases | 17.263 | negativo: 650 (3,8 %); neutral: 825 (4,8 %) | ≈ 24 : 1,3 : 1 |
| C. Satisfacción plena | 17.263 | no excelente: 4.109 (23,8 %) | ≈ 3 : 1 |

**Implicaciones para la modelación.**
- En el escenario A, un clasificador que prediga siempre «positivo» alcanzaría 96 % de exactitud sin aprender nada. Por eso las métricas principales serán F1-macro, exactitud balanceada y *recall* de la clase negativa, y se usará `class_weight='balanced'`.
- Las cifras difieren ligeramente de las del anteproyecto (682 → 650 negativas), porque este corpus excluye los registros `#NAME?` y las reseñas que no están en inglés.